In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd

from linreg.preprocessing import (
    NUMERICAL_COLS, split_by_date, fit_one_hot, transform_one_hot, fit_scaler, transform_scaler,
)
from linreg.models import LinearRegressionNormal

df = pd.read_csv("../data/processed/bart_daily.csv", parse_dates=["date"])

train, test = split_by_date(df)

categories = fit_one_hot(train)
train_enc = transform_one_hot(train, categories)
test_enc = transform_one_hot(test, categories)

stats = fit_scaler(train_enc)
train_ready = transform_scaler(train_enc, stats)
test_ready = transform_scaler(test_enc, stats)

print(train_ready.shape, test_ready.shape)

(2928, 29) (2920, 29)


In [ ]:
feature_cols = [c for c in train_ready.columns if c not in ("date", "exits")]

X_train = train_ready[feature_cols].to_numpy(dtype=float)
y_train = train_ready["exits"].to_numpy(dtype=float)
X_test = test_ready[feature_cols].to_numpy(dtype=float)
y_test = test_ready["exits"].to_numpy(dtype=float)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(2928, 27) (2928,) (2920, 27) (2920,)


In [ ]:
model = LinearRegressionNormal().fit(X_train, y_train)
preds = model.predict(X_test)

print("coef_ shape:", model.coef_.shape)
print("intercept_:", round(model.intercept_))
print("preds shape:", preds.shape)

coef_ shape: (27,)
intercept_: 3698
preds shape: (2920,)


In [6]:
from sklearn.linear_model import LinearRegression

sk = LinearRegression().fit(X_train, y_train)

print("Intercepts match:", np.isclose(model.intercept_, sk.intercept_))
print("Coefficients match:", np.allclose(model.coef_, sk.coef_))
print("Predictions match:", np.allclose(preds, sk.predict(X_test)))

Intercepts match: True
Coefficients match: True
Predictions match: True


In [7]:
coefs = pd.Series(model.coef_, index=feature_cols).round()
coefs.sort_values()

is_holiday              -4503.0
day_of_week_Sunday      -3909.0
day_of_week_Saturday    -2496.0
precip_in                -184.0
month_12                  -88.0
month_7                    63.0
temp_max_f                102.0
day_of_week_Monday        254.0
month_11                  286.0
month_4                   406.0
month_8                   437.0
month_3                   496.0
month_5                   546.0
month_2                   570.0
month_6                   671.0
station_24TH              760.0
station_DBRK              792.0
station_16TH              917.0
month_10                  986.0
month_9                  1047.0
day_of_week_Thursday     1916.0
day_of_week_Tuesday      2017.0
day_of_week_Wednesday    2080.0
station_CIVC             3490.0
station_POWL             5900.0
station_MONT             6843.0
station_EMBR             9696.0
dtype: float64